# Vertragsaufteilung und Frequenz-Basismodell

Wie lassen sich Modelle vergleichen, wenn Verträge unterschiedlich lange beobachtet werden? Die feste Aufteilung trennt die Modellentwicklung von der gemeinsamen Evaluation. Ein Exponierungs-Intercept zeigt, wie Out-of-Fold-Vorhersagen und Poisson-Deviance je Vertragsjahr berechnet werden. [Validierung und Analysegeschichte](../docs/validierung.md)

In [1]:
from pathlib import Path
import json
from hashlib import sha256

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/data_contract.json").is_file())

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def digest(path):
    return sha256(path.read_bytes()).hexdigest()

reports = ROOT / "reports/s2"
split_report = read_json(reports / "split_manifest.json")
baseline_path = reports / "baseline_oof_report.json"
run_report = read_json(baseline_path)
diagnostics = read_json(reports / "baseline_diagnostics.json")
assert run_report["split_id"] == split_report["split_id"]
if diagnostics["baseline_report_sha256"] == digest(baseline_path):
    diagnostics_tables = {}
    for name, table in diagnostics["tables"].items():
        spec = run_report["artifacts"][name]
        assert table["parquet_sha256"] == spec["sha256"]
        assert len(table["rows"]) == spec["rows"]
        diagnostics_tables[name] = pd.DataFrame(table["rows"])
else:
    # A fresh statistical reproduction has its own verified local run.
    import sys
    sys.path.insert(0, str(ROOT))
    from src.validation import read_artifacts

    diagnostics_tables, cached = read_artifacts(ROOT / "data/validation" / run_report["run_id"])
    assert cached == run_report


## Verträge im Entwicklungs- und Evaluationsbestand

Alle Schadenzeilen folgen der Zuordnung ihres Vertrags. Die fünf OOF-Folds teilen nur den Entwicklungsbestand. Vor der heutigen Aufteilung wurde der Gesamtbestand exploriert. Die gemeinsame Evaluation prüft die später festgelegten Modelle innerhalb derselben historischen Datenquelle.

In [2]:
display(pd.DataFrame([
    {"Bestand": "Entwicklung" if name == "development" else "Evaluation", "Verträge": count}
    for name, count in split_report["membership_sizes"].items()
]))
display(pd.DataFrame([
    {"OOF-Fold": fold, "Verträge": count}
    for fold, count in split_report["development_fold_sizes"].items()
]))

,Bestand,Verträge
0,Entwicklung,542392
1,Evaluation,135599


,OOF-Fold,Verträge
0,1,108479
1,2,108479
2,3,108478
3,4,108478
4,5,108478


## Poisson-Deviance je Vertragsjahr

Für einen Vertrag mit Jahresrate $r$ und Exponierung $e$ lautet die erwartete Periodenanzahl $er$. Die Summe der Poisson-Deviance für Periodenanzahlen, geteilt durch die Gesamtexponierung, entspricht der exponierungsgewichteten Deviance der Jahresraten. Das Beispiel verwendet zwei Verträge mit 0,1 und 1,0 Vertragsjahren.

In [3]:
import sys
from sklearn.metrics import mean_poisson_deviance

sys.path.insert(0, str(ROOT))
from src.metrics import poisson_deviance_per_exposure
n_example = np.array([1, 0])
e_example = np.array([0.1, 1.0])
r_example = np.array([1.0, 1.0])
count_score = poisson_deviance_per_exposure(n_example, r_example, e_example)
rate_score = mean_poisson_deviance(n_example / e_example, r_example, sample_weight=e_example)
assert np.isclose(count_score, rate_score, rtol=1e-12)
display(pd.DataFrame({"Berechnung": ["Anzahl-Deviance / Exponierung", "Exponierungsgewichtete Raten-Deviance"],
                      "Score": [count_score, rate_score]}))

,Berechnung,Score
0,Anzahl-Deviance / Exponierung,4.368337
1,Exponierungsgewichtete Raten-Deviance,4.368337


## Frequenz und Kalibrierung der Entwicklungs-OOF-Vorhersagen

Pro Fold schätzt das Basismodell eine gemeinsame Jahresrate aus den übrigen vier Folds. Jeder Entwicklungsvertrag erhält dadurch genau eine Vorhersage aus einem Fit ohne diesen Vertrag.

In [4]:
assert run_report["evaluation_predictions"] == run_report["evaluation_scores"] == 0
summary_labels = {
    "policies": "Entwicklungsverträge", "exposure_years": "Exponierung [Vertragsjahre]",
    "observed_claims": "Beobachtete positive Schadenzeilen", "expected_claims": "Erwartete Periodenanzahl",
    "observed_annual_rate": "Beobachtete Jahresrate", "predicted_annual_rate": "Vorhergesagte Jahresrate",
    "actual_over_predicted": "A/P: tatsächlich / vorhergesagt",
    "poisson_deviance_per_exposure": "Poisson-Deviance je Vertragsjahr",
    "mean_count_deviance": "Mittlere Anzahl-Deviance je Vertragszeile"
}
display(pd.DataFrame([
    {"Größe": summary_labels[name], "Entwicklungswert": value}
    for name, value in run_report["summary"].items()
]))

,Größe,Entwicklungswert
0,A/P: tatsächlich / vorhergesagt,0.999994
1,Erwartete Periodenanzahl,21066.129976
2,Exponierung [Vertragsjahre],286880.818827
3,Mittlere Anzahl-Deviance je Vertragszeile,0.252872
4,Beobachtete Jahresrate,0.073431
5,Beobachtete positive Schadenzeilen,21066.000000
6,Poisson-Deviance je Vertragsjahr,0.478094
7,Entwicklungsverträge,542392.000000
8,Vorhergesagte Jahresrate,0.073432


## Poisson-Deviance je Vertragsjahr nach OOF-Fold

Der gepoolte Score gewichtet die Fold-Scores mit den jeweiligen Vertragsjahren. Das entspricht der Berechnung aus allen einzelnen OOF-Vorhersagen.

In [5]:
folds = pd.DataFrame(run_report["fold_diagnostics"])
weighted_fold_score = np.average(folds["deviance_per_exposure"], weights=folds["validation_exposure_years"])
assert np.isclose(weighted_fold_score, run_report["summary"]["poisson_deviance_per_exposure"], rtol=1e-12)
fold_columns = {
    "fold": "OOF-Fold", "training_policies": "Trainingsverträge", "validation_policies": "Validierungsverträge",
    "training_exposure_years": "Training [Vertragsjahre]", "training_claims": "Trainingsschäden",
    "validation_exposure_years": "Validierung [Vertragsjahre]", "validation_claims": "Validierungsschäden",
    "deviance_per_exposure": "Deviance je Vertragsjahr"
}
display(folds[list(fold_columns)].rename(columns=fold_columns))
print(f"Gepoolte OOF-Deviance je Vertragsjahr: {weighted_fold_score:.6f}")

,OOF-Fold,Trainingsverträge,Validierungsverträge,Training [Vertragsjahre],Trainingsschäden,Validierung [Vertragsjahre],Validierungsschäden,Deviance je Vertragsjahr
0,1,433913,108479,229457.067656,16772,57423.751171,4294,0.484377
1,2,433913,108479,229771.120547,16848,57109.698280,4218,0.481616
2,3,433914,108478,229219.826121,16892,57660.992706,4174,0.470688
3,4,433914,108478,229595.986958,16835,57284.831869,4231,0.479147
4,5,433914,108478,229479.274026,16917,57401.544801,4149,0.474692


Gepoolte OOF-Deviance je Vertragsjahr: 0.478094


## Kalibrierung und Konzentration im Entwicklungsbestand

A/P teilt die beobachtete durch die erwartete Schadenanzahl. Der rohe OOF-Gini kann beim Intercept leicht von null abweichen, weil sich die fünf Trainingsraten unterscheiden. Innerhalb eines Folds weist der Intercept allen Verträgen dieselbe Jahresrate zu.

In [6]:
calibration = diagnostics_tables["calibration"]
assert calibration["policies"].sum() == run_report["summary"]["policies"]
assert calibration["observed_claims"].sum() == run_report["summary"]["observed_claims"]
assert np.isclose(calibration["exposure_years"].sum(), run_report["summary"]["exposure_years"])
calibration_labels = {
    "group": "Gruppe", "min_annual_rate": "Kleinste Jahresrate", "max_annual_rate": "Größte Jahresrate",
    "policies": "Verträge", "exposure_years": "Exponierung [Vertragsjahre]",
    "observed_claims": "Beobachtete Anzahlen", "expected_claims": "Erwartete Anzahlen",
    "observed_annual_rate": "Beobachtete Jahresrate", "predicted_annual_rate": "Vorhergesagte Jahresrate",
    "actual_over_predicted": "A/P"
}
display(calibration.rename(columns=calibration_labels))
curve = diagnostics_tables["concentration"][
    ["cumulative_exposure_fraction", "cumulative_claim_fraction"]
]
assert np.allclose(curve.iloc[0].to_numpy(), [0, 0])
assert np.allclose(curve.iloc[-1].to_numpy(), [1, 1])
raw_gini = 2 * np.trapezoid(curve["cumulative_claim_fraction"], curve["cumulative_exposure_fraction"]) - 1
display(curve.rename(columns={"cumulative_exposure_fraction": "Kumulierte Exponierung",
                             "cumulative_claim_fraction": "Kumulierte beobachtete Anzahlen"}))
print(f"Roher gepoolter OOF-Gini des Basismodells: {raw_gini:.6f}")

,A/P,Erwartete Anzahlen,Exponierung [Vertragsjahre],Gruppe,Größte Jahresrate,Kleinste Jahresrate,Beobachtete Jahresrate,Beobachtete Anzahlen,Verträge,Vorhergesagte Jahresrate
0,1.023027,4197.347959,57423.751171,2,0.073094,0.073094,0.074777,4294,108479,0.073094
1,1.007290,4200.378923,57284.831869,4,0.073324,0.073324,0.073859,4231,108478,0.073324
2,1.007265,4187.576726,57109.698280,6,0.073325,0.073325,0.073858,4218,108479,0.073325
3,0.982294,4249.237534,57660.992706,8,0.073693,0.073693,0.072389,4174,108478,0.073693
4,0.980483,4231.588833,57401.544801,10,0.073719,0.073719,0.072280,4149,108478,0.073719


,Kumulierte Exponierung,Kumulierte beobachtete Anzahlen
0,0.000000,0.000000
1,0.200088,0.196952
2,0.401081,0.395092
3,0.600152,0.595319
4,0.799834,0.796164
5,1.000000,1.000000


Roher gepoolter OOF-Gini des Basismodells: -0.007049


## Referenz für den Modellvergleich

Der Intercept liefert den Vergleichswert für Modelle, die Risiken nach Vertragsmerkmalen unterscheiden. [GLM, GAM und Boosting](04_frequency_models.ipynb) werden auf denselben Entwicklungs-OOF-Verträgen beurteilt. Die [gemeinsame Evaluation](05_severity_pure_premium.ipynb) verwendet die davor festgelegten Regeln.